# Gradient Analysis

Two diagnostic experiments for Section 4 (Frequency Reachability):
1. **Prefactor displacement** — train unary circuit on Ω₂ across three learning rates,
   save full alpha history → `fig:grad_prefactor_analysis` (a) and appendix evolution plots
2. **Gradient sweep** — measure gradient magnitude at initialization across a sweep
   of initial prefactor values → `fig:grad_prefactor_analysis` (b)

In [ ]:
# ── Smoke test flag ────────────────────────────────────────────
SMOKE_TEST = False

from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pennylane as qml
import pandas as pd
import pickle
import sys
import re
from sklearn.metrics import r2_score

sys.path.append("..")
from paper_style import apply_paper_style, BLUE, RED, GREEN, ORANGE, PURPLE
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)
apply_paper_style()
SEED = 42
np.random.seed(SEED)

## Hyperparameters

In [ ]:
NUM_FUNCTIONS  = 1 if SMOKE_TEST else 10
NUM_RUNS       = 2 if SMOKE_TEST else 10
MAX_STEPS      = 50  if SMOKE_TEST else 5000
BATCH_SIZE     = 40
NUM_WIRES      = 3
NUM_SERIAL_LAYERS  = 2
TRAINABLE_BLOCKS   = 1
NUM_SU_GATES       = 1
NUM_ROT_PARAMS     = 63

# Initial prefactors for displacement experiment
ALPHA_INIT  = [[1.01, 1.02, 1.03]]  # perturbed unary
LR_VALUES   = [0.001, 0.01, 0.1]

# Sweep range for gradient locality experiment
SWEEP_MIN   = 1
SWEEP_MAX   = 25 if SMOKE_TEST else 25

DATA_DIR = Path("../datasets")
OUT_DIR  = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"SMOKE_TEST={SMOKE_TEST}  functions={NUM_FUNCTIONS}  runs={NUM_RUNS}")

## Circuit definition

In [ ]:
def S(alpha, x, num_wires, enc_layer):
    for w in range(num_wires):
        qml.RX(alpha[w] * x, wires=w)

def W_SU(theta, trainable_block_layers, num_wires):
    for i in range(trainable_block_layers):
        qml.SpecialUnitary(theta[i][0], wires=range(num_wires))

def weights_ones(n_serial, t_blocks, n_su, n_params):
    return np.ones((n_serial, t_blocks, n_su, n_params))

def square_loss(targets, predictions):
    return 0.5 * sum((t-p)**2 for t,p in zip(targets, predictions)) / len(targets)

dev = qml.device("default.qubit", wires=NUM_WIRES)

@qml.qnode(dev, interface="jax")
def circuit(alpha, weights_SU, x=None):
    W_SU(weights_SU[0], TRAINABLE_BLOCKS, NUM_WIRES)
    for i in range(NUM_SERIAL_LAYERS - 1):
        S(alpha[i], x, NUM_WIRES, i)
        W_SU(weights_SU[i+1], TRAINABLE_BLOCKS, NUM_WIRES)
    return qml.expval(qml.PauliZ(wires=NUM_WIRES-1))

circuit_jit = jax.jit(circuit)

## Load data

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg_10.pkl", "rb") as f:
    datasets = pickle.load(f)
print(f"Loaded {len(datasets)} functions (Ω₂)")

## Experiment 1: Prefactor displacement across learning rates

In [ ]:
displacement_results = []

for lr in LR_VALUES:
    for func in range(NUM_FUNCTIONS):
        ds = datasets[func]
        x_tr, y_tr = ds["x_train"], ds["y_train"]
        x_te, y_te = ds["x_test"],  ds["y_test"]

        for run in range(NUM_RUNS):
            alpha   = jnp.array(ALPHA_INIT)
            weights = jnp.array(weights_ones(
                NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS,
                NUM_SU_GATES, NUM_ROT_PARAMS
            ))
            params    = {"weights_SU": weights, "alpha": alpha}
            opt       = optax.adam(lr)
            opt_state = opt.init(params)

            # Record alpha at each step for evolution plot
            alpha_history = [alpha.tolist()]

            def cost_fn(p, x, y):
                preds = [circuit_jit(p["alpha"], p["weights_SU"], x_)
                         for x_ in x]
                return square_loss(y, preds).squeeze()

            @jax.jit
            def update(p, s, x, y):
                loss, g = jax.value_and_grad(cost_fn)(p, x, y)
                u, s2   = opt.update(g, s, p)
                return optax.apply_updates(p, u), s2, loss

            for step in range(MAX_STEPS):
                idx = np.random.randint(0, len(x_tr), BATCH_SIZE)
                params, opt_state, c = update(
                    params, opt_state,
                    jnp.array(x_tr[idx]), jnp.array(y_tr[idx])
                )
                if (step + 1) % 100 == 0:
                    alpha_history.append(params["alpha"].tolist())

            final_alpha = params["alpha"]
            displacement = float(jnp.max(jnp.abs(
                final_alpha - jnp.array(ALPHA_INIT)
            )))

            preds = [float(circuit_jit(final_alpha, params["weights_SU"], x_))
                     for x_ in x_te]
            r2 = r2_score(y_te, preds)

            # Save alpha history as .npy for evolution plot
            hist_path = OUT_DIR / f"alpha_hist_lr{lr}_func{func}_run{run}.npy"
            np.save(hist_path, np.array(alpha_history))

            displacement_results.append({
                "lr":           lr,
                "Func":         func,
                "Run":          run,
                "R2_Score":     r2,
                "displacement": displacement,
                "final_alpha":  final_alpha.tolist(),
                "alpha_hist_file": str(hist_path)
            })
            print(f"  lr={lr}  func={func+1}  run={run+1}  "
                  f"disp={displacement:.3f}  R²={r2:.4f}")

disp_df = pd.DataFrame(displacement_results)
disp_df.to_csv(OUT_DIR / "displacement_results.csv", index=False)
print("Displacement experiment saved.")

## Experiment 2: Gradient sweep across initial prefactor values

In [ ]:
sweep_results = []
sweep_values  = np.arange(SWEEP_MIN, SWEEP_MAX + 1, dtype=float)
weights_init  = jnp.array(weights_ones(
    NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS, NUM_SU_GATES, NUM_ROT_PARAMS
))

for alpha_val in sweep_values:
    alpha = jnp.array([[alpha_val, alpha_val, alpha_val]])

    for func in range(NUM_FUNCTIONS):
        ds = datasets[func]
        x_tr, y_tr = ds["x_train"], ds["y_train"]

        for run in range(NUM_RUNS):
            params = {"weights_SU": weights_init, "alpha": alpha}
            opt    = optax.adam(0.001)

            def cost_fn(p, x, y):
                preds = [circuit_jit(p["alpha"], p["weights_SU"], x_)
                         for x_ in x]
                return square_loss(y, preds).squeeze()

            # Single gradient evaluation at initialization
            idx    = np.random.randint(0, len(x_tr), BATCH_SIZE)
            x_b    = jnp.array(x_tr[idx])
            y_b    = jnp.array(y_tr[idx])
            _, grads = jax.value_and_grad(cost_fn)(params, x_b, y_b)

            sweep_results.append({
                "prefactors": alpha_val,
                "Func":       func,
                "Run":        run,
                "gradients":  str(grads["alpha"].tolist())
            })

    print(f"  Sweep α={alpha_val:.0f} done")

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(OUT_DIR / "gradient_sweep.csv", index=False)
print("Gradient sweep saved.")

## Plot — fig:grad_prefactor_analysis

In [ ]:
disp_df  = pd.read_csv(OUT_DIR / "displacement_results.csv")
sweep_df = pd.read_csv(OUT_DIR / "gradient_sweep.csv")

# ── Parse gradient magnitudes from sweep ──
def parse_grad_mag(s):
    s_clean = re.sub(r'dtype=\S+', '', str(s))
    nums = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?', s_clean)
    vals = [float(x) for x in nums]
    return max(abs(v) for v in vals) if vals else np.nan

sweep_df["grad_abs_max"] = sweep_df["gradients"].apply(parse_grad_mag)

grad_mean = sweep_df.groupby("prefactors")["grad_abs_max"].mean()
grad_std  = sweep_df.groupby("prefactors")["grad_abs_max"].std()
pf_vals   = grad_mean.index.values

# ── Panel (a): displacement violins ──
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))

lr_colors = [BLUE, ORANGE, RED]
for ax_lr, (lr, col) in zip([], zip(LR_VALUES, lr_colors)):
    pass  # handled below

ax = axes[0]
data_by_lr = {
    rf"$\eta={lr}$": disp_df[disp_df.lr==lr]["displacement"].values
    for lr in LR_VALUES
}
positions = np.arange(1, len(LR_VALUES)+1)
parts = ax.violinplot(
    list(data_by_lr.values()), positions=positions,
    showmedians=True, showextrema=False, widths=0.6
)
for pc, col in zip(parts["bodies"], lr_colors):
    pc.set_facecolor(col); pc.set_alpha(0.70); pc.set_edgecolor("black"); pc.set_linewidth(0.8)
parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(2.0)
ax.set_xticks(positions)
ax.set_xticklabels(list(data_by_lr.keys()), fontsize=8)
ax.set_ylabel(r"Max prefactor displacement", fontsize=9, fontweight="bold")
ax.grid(True, alpha=0.5, linestyle="-", linewidth=0.8)
ax.set_axisbelow(True)

# ── Panel (b): gradient sweep ──
ax = axes[1]
ax.plot(pf_vals, grad_mean.values, color=BLUE, linewidth=1.5,
        label=r"Mean $|\partial\mathcal{L}/\partial\alpha_i|$")
ax.fill_between(pf_vals,
                grad_mean.values - grad_std.values,
                grad_mean.values + grad_std.values,
                alpha=0.25, color=BLUE)

# Mark target frequencies
for tfreq in [11, 11.2, 13]:
    ax.axvline(tfreq, color=RED, linestyle="--", linewidth=0.9, alpha=0.8)
ax.axvline(11, color=RED, linestyle="--", linewidth=0.9, alpha=0.8,
           label=r"Target frequencies")

ax.set_xlabel(r"Prefactor value", fontsize=9, fontweight="bold")
ax.set_ylabel(r"Gradient magnitude", fontsize=9, fontweight="bold")
ax.legend(loc="upper left", frameon=True, fancybox=False,
          shadow=False, fontsize=7)
ax.grid(True, alpha=0.5, linestyle="-", linewidth=0.8)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("grad_prefactor_analysis.pdf", dpi=600, bbox_inches="tight")
plt.savefig("grad_prefactor_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved grad_prefactor_analysis.pdf")

## Appendix plot: prefactor evolution by learning rate
(best run per learning rate — used in appendix fig:prefactor_evolution)

In [ ]:
pf_colors = [BLUE, RED, GREEN]
labels    = [r"$\alpha_1$", r"$\alpha_2$", r"$\alpha_3$"]

for lr_val, lr_col in zip(LR_VALUES, lr_colors):
    sub      = disp_df[disp_df.lr == lr_val]
    best_row = sub.loc[sub.R2_Score.idxmax()]
    hist     = np.load(best_row["alpha_hist_file"]).squeeze()
    epochs   = np.arange(len(hist)) * 100

    fig, ax = plt.subplots(figsize=(3.3, 2.5))
    for i, (lbl, col) in enumerate(zip(labels, pf_colors)):
        ax.plot(epochs, hist[:, i], label=lbl, color=col,
                linewidth=1.5, alpha=0.85)
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.9,
               alpha=0.6, label="Initial value")
    ax.set_xlabel(r"Step", fontsize=9, fontweight="bold")
    ax.set_ylabel(r"Prefactor value", fontsize=9, fontweight="bold")
    ax.legend(frameon=True, fancybox=False, fontsize=7)
    ax.grid(True, alpha=0.5, linestyle="-", linewidth=0.8)
    plt.tight_layout()
    plt.savefig(f"prefactor_evolution_lr{lr_val}.pdf",
                dpi=600, bbox_inches="tight")
    plt.show()
    print(f"Saved prefactor_evolution_lr{lr_val}.pdf")